# Ad Auction Simulator — Full Analysis

Полный пайплайн: генерация данных → аукцион → CTR-модель → оптимизация ставок → A/B тест

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Генерация данных

In [ ]:
from data.generate import generate_advertisers, generate_queries, generate_impressions

advertisers = generate_advertisers(50)
queries = generate_queries(10_000)
impressions = generate_impressions(advertisers, queries)

print(f'Advertisers: {len(advertisers)}')
print(f'Queries:     {len(queries)}')
print(f'Impressions: {len(impressions)}')
impressions.head()

## 2. EDA — Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# CTR по позициям
ctr_by_rank = impressions.groupby('rank')['click'].mean()
axes[0].bar(ctr_by_rank.index, ctr_by_rank.values, color='steelblue')
axes[0].set_title('CTR by Position')
axes[0].set_xlabel('Rank')
axes[0].set_ylabel('CTR')

# Распределение ставок
axes[1].hist(impressions['bid'], bins=40, color='coral', edgecolor='white')
axes[1].set_title('Bid Distribution')
axes[1].set_xlabel('Bid')

# CTR по категориям
ctr_by_cat = impressions.groupby('category')['click'].mean().sort_values()
axes[2].barh(ctr_by_cat.index, ctr_by_cat.values, color='mediumseagreen')
axes[2].set_title('CTR by Category')

plt.tight_layout()
plt.show()

## 3. Аукционные механизмы: GSP vs VCG

In [ ]:
from auction import Bid, run_gsp, run_vcg, compare_mechanisms
import numpy as np

rng = np.random.default_rng(42)
n_auctions = 1000
gsp_revenues, vcg_revenues = [], []

for _ in range(n_auctions):
    bids = [Bid(i, rng.uniform(5, 100), rng.uniform(0.5, 1.5)) for i in range(10)]
    gsp_rev = sum(r.price_paid for r in run_gsp(bids))
    vcg_rev = sum(r.price_paid for r in run_vcg(bids))
    gsp_revenues.append(gsp_rev)
    vcg_revenues.append(vcg_rev)

print(f'GSP avg revenue: {np.mean(gsp_revenues):.2f}')
print(f'VCG avg revenue: {np.mean(vcg_revenues):.2f}')

plt.figure(figsize=(8, 4))
plt.hist(gsp_revenues, bins=40, alpha=0.6, label='GSP', color='steelblue')
plt.hist(vcg_revenues, bins=40, alpha=0.6, label='VCG', color='coral')
plt.legend()
plt.title('Revenue Distribution: GSP vs VCG')
plt.xlabel('Revenue per Auction')
plt.show()

## 4. CTR-модель

In [ ]:
from models.ctr_model import train_and_compare
lr_model, nn_model = train_and_compare(impressions)

## 5. Оптимизация ставок

In [ ]:
from models.bid_optimizer import (
    analytical_optimal_bid, GradientBidOptimizer,
    BidOptimizerConfig, simulate_budget_pacing
)

# Аналитическая оптимальная ставка
predicted_ctr = 0.05
target_cpc = 30.0
opt_bid = analytical_optimal_bid(predicted_ctr, target_cpc)
print(f'Analytical optimal bid: {opt_bid}')

# Градиентная оптимизация
config = BidOptimizerConfig(budget=5000, target_cpc=30.0)
competitor_bids = rng.uniform(10, 80, 9)
optimizer = GradientBidOptimizer(config, competitor_bids)
grad_bid = optimizer.optimize(predicted_ctr)
print(f'Gradient optimal bid:   {grad_bid}')

# Симуляция расхода бюджета
pacing = simulate_budget_pacing(
    bids_per_round=[opt_bid] * 100,
    ctrs=[predicted_ctr] * 100,
    budget=5000,
    n_rounds=1000,
)
print('Budget pacing:', pacing)

## 6. A/B тесты

In [ ]:
from experiments.ab_test import (
    run_position_experiment,
    run_mechanism_revenue_experiment,
    run_bid_strategy_experiment,
)

print()
run_position_experiment(impressions)

print()
run_mechanism_revenue_experiment(np.array(gsp_revenues), np.array(vcg_revenues))

print()
# Симулируем клики для двух стратегий ставок
baseline_clicks = rng.binomial(1, 0.03, 500).astype(float)
optimized_clicks = rng.binomial(1, 0.05, 500).astype(float)
run_bid_strategy_experiment(baseline_clicks, optimized_clicks)

## 7. SQL-аналитика

In [ ]:
engine = create_engine('sqlite:///../data/auction.db')
impressions.to_sql('impressions', engine, if_exists='replace', index=False)
advertisers.to_sql('advertisers', engine, if_exists='replace', index=False)

query = """
SELECT
    i.category,
    i.rank,
    COUNT(*) AS impressions,
    SUM(i.click) AS clicks,
    ROUND(AVG(i.click), 4) AS ctr,
    ROUND(AVG(i.bid), 2) AS avg_bid
FROM impressions i
GROUP BY i.category, i.rank
ORDER BY i.category, i.rank
"""

pd.read_sql(query, engine)